# 02 — Judge Scoring

Run all 50 evaluation instances through each of the three Ollama judge models and record
the raw scores.

**Prerequisites:**
- Notebook 01 has been run (answer corpus and human scores exist in `data/`)
- `ollama` is installed (`make setup` handles this)
- Models are pulled: `make setup` pulls `qwen2.5:1.5b`, `gemma3:1b`, `llama3.2:1b`

> **Do NOT run `ollama serve` manually.** This notebook starts and restarts Ollama
> automatically. A parallel terminal instance causes "address already in use" errors.

**Outputs produced:**
- `data/eval/scores_qwen2_5_1_5b.json`
- `data/eval/scores_gemma3_1b.json`
- `data/eval/scores_llama3_2_1b.json`

> **RAM constraint:** Models are scored sequentially. All three models are under 1.5 GB —
> do not run two judges simultaneously in the same Codespace.

In [1]:
import math
import os
import socket
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import httpx
import json
import time

import pandas as pd

from src.config import load_settings
from src.judging.judge import OllamaJudge
from src.judging.runner import METRICS

settings = load_settings(ROOT / "config" / "settings.yaml")
print(f"Ollama URL: {settings.ollama_url}")
print(f"Models: {settings.models}")

FIGURES_DIR = ROOT / "outputs" / "figures"
RESULTS_DIR = ROOT / "outputs" / "results"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# ── helpers ───────────────────────────────────────────────────────────────────

def _nan_count(records: list) -> int:
    return sum(1 for r in records if isinstance(r.get("score"), float) and math.isnan(r["score"]))


def _port_free(port: int = 11434) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


def drop_page_cache() -> None:
    """Release Linux page/slab/dentry caches (the buff/cache shown in free -m)."""
    result = subprocess.run(
        ["sudo", "sh", "-c", "sync; echo 3 > /proc/sys/vm/drop_caches"],
        capture_output=True,
    )
    if result.returncode == 0:
        print("  Page cache dropped.")
    else:
        print(f"  drop_caches skipped (no sudo or not supported): {result.stderr.decode().strip()}")


def ensure_ollama_running(ollama_url: str, timeout: int = 60) -> bool:
    """Start Ollama if it is not already running. Returns True when ready."""
    if not _port_free():
        return True  # already running
    print("Ollama not running — starting ollama serve...", end=" ", flush=True)
    env = {**os.environ, "OLLAMA_NUM_PARALLEL": "1", "OLLAMA_MAX_LOADED_MODELS": "1"}
    subprocess.Popen(
        ["ollama", "serve"],
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            httpx.get(f"{ollama_url}/api/tags", timeout=3.0)
            print("ready.")
            return True
        except Exception:
            time.sleep(2)
    print("WARNING: Ollama did not start within timeout.")
    return False


def restart_ollama(ollama_url: str, timeout: int = 60) -> None:
    """Kill Ollama + its llama runner child, drop page cache, start fresh."""
    print("  Restarting Ollama...", end=" ", flush=True)
    subprocess.run(["pkill", "-TERM", "-f", "ollama serve"], capture_output=True)
    for _ in range(20):  # wait up to 10 s for port to be released
        if _port_free():
            break
        time.sleep(0.5)
    else:
        subprocess.run(["pkill", "-KILL", "-f", "ollama serve"], capture_output=True)
        time.sleep(1)
    # Kill orphaned llama runner — it holds model weights in RAM for several
    # seconds after the Ollama server exits, causing OOM when the next model loads.
    subprocess.run(["pkill", "-KILL", "-f", "ollama_llama_server"], capture_output=True)
    time.sleep(4)  # let runner fully exit and release RAM before dropping cache
    drop_page_cache()
    env = {**os.environ, "OLLAMA_NUM_PARALLEL": "1", "OLLAMA_MAX_LOADED_MODELS": "1"}
    subprocess.Popen(
        ["ollama", "serve"],
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            httpx.get(f"{ollama_url}/api/tags", timeout=3.0)
            print("ready.")
            return
        except Exception:
            time.sleep(2)
    print("WARNING: Ollama did not restart within timeout.")


Ollama URL: http://localhost:11434
Models: ['llama3.2:1b', 'gemma3:1b', 'qwen2.5:1.5b']


## 1. Load answer corpus

In [2]:
answers_dir = ROOT / "data" / "answers"

all_instances = []
for path in sorted(answers_dir.glob("*.json")):
    data = json.loads(path.read_text())
    if isinstance(data, list):
        all_instances.extend(data)
    else:
        all_instances.append(data)

print(f"Loaded {len(all_instances)} instances from {answers_dir}")
pd.DataFrame(all_instances).groupby(["pipeline", "question_type"]).size().rename("count").reset_index()

Loaded 50 instances from /workspaces/llm_judge_benchmark/data/answers


,pipeline,question_type,count
0,graph,absence_reasoning,5
1,graph,multi_hop,6
2,graph,single_hop,9
3,handcrafted,absence_reasoning,3
4,handcrafted,multi_hop,3
5,handcrafted,single_hop,4
6,vector,absence_reasoning,5
7,vector,multi_hop,6
8,vector,single_hop,9


## 2. Check Ollama availability

In [3]:
ensure_ollama_running(settings.ollama_url)

try:
    resp = httpx.get(f"{settings.ollama_url}/api/tags", timeout=5.0)
    available_models = [m["name"] for m in resp.json().get("models", [])]
    print(f"Ollama is running. Available models: {available_models}")
except Exception as e:
    print(f"Ollama not reachable: {e}")
    print("Check that 'ollama' is installed: curl -fsSL https://ollama.com/install.sh | sh")
    available_models = []


Ollama not running — starting ollama serve... ready.
Ollama is running. Available models: ['llama3.2:1b', 'gemma3:1b', 'qwen2.5:1.5b']


## 3. Score all instances

Each model is scored independently. Results are written to `data/eval/` after each model
so progress survives interruptions. Ollama is restarted before each model for a clean
memory slate — no manual terminal commands needed.

Estimated times on a 2-CPU Codespace (includes Ollama restart overhead):
- `qwen2.5:1.5b`: ~15 min
- `gemma3:1b`: ~12 min
- `llama3.2:1b`: ~10 min

In [4]:
# ── Configuration ──────────────────────────────────────────────────────────────
# FORCE_RESCORE = True  → always re-score, even if a valid file exists.
# FORCE_RESCORE = False → skip models with complete, NaN-free score files only.
#                         Models with missing files OR any NaN scores are re-scored automatically.
FORCE_RESCORE = True # False

eval_dir = ROOT / "data" / "eval"
eval_dir.mkdir(parents=True, exist_ok=True)

print("Score file status:\n")
for model in settings.models:
    safe = model.replace(":", "_").replace(".", "_")
    out_path = eval_dir / f"scores_{safe}.json"
    if out_path.exists():
        records = json.loads(out_path.read_text())
        nans = _nan_count(records)
        if not records:
            tag, detail, action = "[EMPTY]", "0 records", "will score"
        elif nans > 0:
            tag = "[NaN]"
            detail = f"{len(records)} records, {nans} NaN"
            action = "will RE-SCORE (NaN found)" if not FORCE_RESCORE else "will RE-SCORE (forced)"
        else:
            tag = "[OK]"
            detail = f"{len(records)} records, 0 NaN"
            action = "SKIP" if not FORCE_RESCORE else "will RE-SCORE (forced)"
    else:
        tag, detail, action = "[MISSING]", "—", "will score"
    print(f"  {tag:<10} {out_path.name:<32} {detail:<26} → {action}")

Score file status:

  [MISSING]  scores_llama3_2_1b.json          —                          → will score
  [MISSING]  scores_gemma3_1b.json            —                          → will score
  [OK]       scores_qwen2_5_1_5b.json         150 records, 0 NaN         → will RE-SCORE (forced)


In [5]:
def score_model(model: str, instances: list, output_dir: Path) -> list:
    """Score all instances with one judge using a single combined call per instance."""
    judge = OllamaJudge(model=model, ollama_url=settings.ollama_url, num_ctx=settings.num_ctx)
    records = []
    total = len(instances)
    t0 = time.perf_counter()

    for inst_idx, instance in enumerate(instances, 1):
        try:
            scores = judge.score_all_metrics(
                question=str(instance["question"]),
                context=instance["context"],
                answer=str(instance["answer"]),
            )
        except Exception as exc:
            print(f"  ERROR {instance['id']}: {exc}")
            scores = {m: float("nan") for m in METRICS}

        for metric, score in scores.items():
            records.append({"id": instance["id"], "model": model, "metric": metric, "score": score})

        elapsed = time.perf_counter() - t0
        rate = inst_idx / elapsed if elapsed > 0 else 0
        eta = (total - inst_idx) / rate if rate > 0 else 0
        score_summary = ", ".join(f"{m[:4]}={scores[m]:.3f}" for m in METRICS)
        print(f"  [{model}] {inst_idx}/{total}: {score_summary}  ({elapsed:.0f}s elapsed, ETA {eta:.0f}s)")

    elapsed = time.perf_counter() - t0
    safe = model.replace(":", "_").replace(".", "_")
    out_path = output_dir / f"scores_{safe}.json"
    out_path.write_text(json.dumps(records, indent=2))
    print(f"  Saved {len(records)} records to {out_path}  ({elapsed:.1f}s total)")
    return records

In [6]:
timing = {}

for model in settings.models:
    safe = model.replace(":", "_").replace(".", "_")
    out_path = eval_dir / f"scores_{safe}.json"

    if not FORCE_RESCORE and out_path.exists():
        existing = json.loads(out_path.read_text())
        nans = _nan_count(existing)
        if existing and nans == 0:
            print(f"SKIP {model}: {len(existing)} valid records in {out_path.name}")
            continue
        reason = f"{nans} NaN scores" if nans > 0 else "empty file"
        print(f"RE-SCORE {model}: {reason} in {out_path.name}, deleting and re-running")
        out_path.unlink()

    if model not in available_models:
        print(f"SKIP {model}: not available in Ollama (pull with: ollama pull {model})")
        continue

    # Restart Ollama for a clean memory slate — avoids state corruption from
    # the previous model and eliminates residual KV-cache allocations.
    restart_ollama(settings.ollama_url)

    # Warm up: load the model and confirm it responds before the scoring loop.
    judge_check = OllamaJudge(model=model, ollama_url=settings.ollama_url, num_ctx=settings.num_ctx)
    if not judge_check.warm_up(timeout=120.0):
        print(f"  SKIP {model}: model failed to load after Ollama restart")
        continue

    print(f"\nScoring with {model} ({len(all_instances)} instances \u00d7 {len(METRICS)} metrics)...")
    t0 = time.perf_counter()
    score_model(model, all_instances, eval_dir)
    timing[model] = time.perf_counter() - t0

if timing:
    print("\n=== Timing summary ===")
    for m, t in timing.items():
        print(f"  {m}: {t/60:.1f} min")
else:
    print("\nAll models skipped (already scored or unavailable).")


  Restarting Ollama...   Page cache dropped.


2026-05-15 07:35:32.023 | INFO     | src.judging.judge:warm_up:85 - llama3.2:1b | warming up (model load may take up to 120s)...


ready.


2026-05-15 07:35:42.335 | INFO     | src.judging.judge:warm_up:99 - llama3.2:1b | warm-up complete, model is ready



Scoring with llama3.2:1b (50 instances × 3 metrics)...


...

  [qwen2.5:1.5b] 48/50: cont=1.000, grou=1.000, answ=1.000  (570s elapsed, ETA 24s)


2026-05-15 08:03:02.292 | DEBUG    | src.judging.judge:score_all_metrics:64 - qwen2.5:1.5b | combined | raw='```json\n{\n  "context_relevance": 0.9,\n  "groundedness": 1.0,\n  "answer_relevance": 0.85\n}\n```' | scores={'context_relevance': 0.9, 'groundedness': 1.0, 'answer_relevance': 0.85}


  [qwen2.5:1.5b] 49/50: cont=0.900, grou=1.000, answ=0.850  (583s elapsed, ETA 12s)


2026-05-15 08:03:13.259 | DEBUG    | src.judging.judge:score_all_metrics:64 - qwen2.5:1.5b | combined | raw='{\n  "context_relevance": 0.6,\n  "groundedness": 0.5,\n  "answer_relevance": 0.8\n}' | scores={'context_relevance': 0.6, 'groundedness': 0.5, 'answer_relevance': 0.8}


  [qwen2.5:1.5b] 50/50: cont=0.600, grou=0.500, answ=0.800  (594s elapsed, ETA 0s)
  Saved 150 records to /workspaces/llm_judge_benchmark/data/eval/scores_qwen2_5_1_5b.json  (594.1s total)

=== Timing summary ===
  llama3.2:1b: 8.3 min
  gemma3:1b: 8.7 min
  qwen2.5:1.5b: 9.9 min


## 4. Score summary

In [7]:
from IPython.display import display

score_files = list(eval_dir.glob("scores_*.json"))
if not score_files:
    print("No score files found. Run the scoring cell above first.")
else:
    all_records = []
    for f in sorted(score_files):
        all_records.extend(json.loads(f.read_text()))

    if not all_records:
        print("Score files found but all are empty — run the scoring cell above.")
    else:
        scores_df = pd.DataFrame(all_records)
        print(f"Total score records: {len(scores_df)}")
        print(f"Models scored: {sorted(scores_df['model'].unique())}")
        print("\nMean score per (model, metric):")
        display(scores_df.groupby(["model", "metric"])["score"].mean().unstack().round(3))

        summary = scores_df.groupby(["model", "metric"])["score"].agg(["mean", "std", "count"]).round(4)
        summary_path = RESULTS_DIR / "02_score_summary.json"
        summary.reset_index().to_json(summary_path, orient="records", indent=2)
        print(f"Saved score summary to {summary_path}")

Total score records: 450
Models scored: ['gemma3:1b', 'llama3.2:1b', 'qwen2.5:1.5b']

Mean score per (model, metric):


metric,answer_relevance,context_relevance,groundedness
model,,,
gemma3:1b,0.866,0.771,0.880
llama3.2:1b,0.645,0.468,0.595
qwen2.5:1.5b,0.881,0.824,0.833


Saved score summary to /workspaces/llm_judge_benchmark/outputs/results/02_score_summary.json


## 5. Load and inspect human scores

In [8]:
human_path = ROOT / "data" / "human" / "human_scores.csv"
if human_path.exists():
    human_df = pd.read_csv(human_path)
    print(f"Human scores: {len(human_df)} instances")
    print("\nMean human scores by metric:")
    print(human_df[["context_relevance", "groundedness", "answer_relevance"]].mean().round(3).to_string())
else:
    print("Human scores not found — run notebook 01 first.")

Human scores: 50 instances

Mean human scores by metric:
context_relevance    0.741
groundedness         0.773
answer_relevance     0.819


---
**Next:** Run `03_inter_judge_agreement.ipynb` to compute kappa and correlation metrics across all annotators.